## Bronze layer Ingestion

In [0]:
account_key=dbutils.secrets.get(scope = "My_scope", key = "DB-secret")
# plx = dbutils.secrets.get(scope = "my_scope", key = "sa")
# print(plx)
# dbutils.secrets.listScopes()

In [0]:
storage_account = "storagestartup"
container = "startup-data"
spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    account_key
)

In [0]:
df = spark.read.csv(
    "abfss://startup-data@storagestartup.dfs.core.windows.net/bronze/startup_funding/indian_startup_funding_2020_2025_sample.csv",
    header=True,
    inferSchema=True
)


In [0]:
print(df.count())

1100


In [0]:
print(len(df.columns))

8


In [0]:
df.printSchema()

root
 |-- Startup: string (nullable = true)
 |-- Industry: string (nullable = true)
 |-- SubVertical: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Investors: string (nullable = true)
 |-- InvestmentType: string (nullable = true)
 |-- InvestmentAmount_USD: integer (nullable = true)
 |-- Date: string (nullable = true)



In [0]:
df.show(5, truncate=False)

+--------+--------------------+------------+---------+-------------------------------+--------------+--------------------+----------+
|Startup |Industry            |SubVertical |City     |Investors                      |InvestmentType|InvestmentAmount_USD|Date      |
+--------+--------------------+------------+---------+-------------------------------+--------------+--------------------+----------+
|Housejoy|EdTech              |K12         |Mumbai   |Lightspeed India               |Seed          |199000              |19-04-2023|
|Groww   |Media               |Streaming   |Bengaluru|IFC                            |Seed          |1668000             |28-01-2025|
|Groww   |Mobility            |Ride Sharing|Hyderabad|Nexus Venture Partners, Peak XV|Series B      |38052000            |14-03-2021|
|FarmBox |Consumer Electronics|Wearables   |Gurugram |Kalaari Capital, Y Combinator  |Seed          |455000              |11/9/2023 |
|Udaan   |RealEstate          |Rental Tech |Mumbai   |Bessemer

In [0]:
df.describe().show()

+-------+-------------+--------+-----------+---------+--------------+--------------+--------------------+--------+
|summary|      Startup|Industry|SubVertical|     City|     Investors|InvestmentType|InvestmentAmount_USD|    Date|
+-------+-------------+--------+-----------+---------+--------------+--------------+--------------------+--------+
|  count|         1100|    1100|       1100|     1100|          1100|          1100|                1100|    1100|
|   mean|         NULL|    NULL|       NULL|     NULL|          NULL|          NULL|2.5532948181818184E7|    NULL|
| stddev|         NULL|    NULL|       NULL|     NULL|          NULL|          NULL| 7.266654908494356E7|    NULL|
|    min|AeroAnalytics|AgriTech|  Analytics|Ahmedabad|  A91 Partners|         Angel|                5000|1/1/2021|
|    max|       Zomato|    SaaS|  Wearables|     Pune|Zodius Capital|      Series D|           574676000|9/9/2024|
+-------+-------------+--------+-----------+---------+--------------+-----------

# Write DF to ADLS as delta table And read back from storage

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("abfss://startup-data@storagestartup.dfs.core.windows.net/bronze/delta/startup_funding")

In [0]:
bronze_df = spark.read.format("delta").load(
    "abfss://startup-data@storagestartup.dfs.core.windows.net/bronze/delta/startup_funding"
)



## Bronze Layer Summary

### Objective
Read raw startup funding data from Azure Data Lake Storage Gen2 and store it as a Delta table without modifying the original data.

### Completed Steps
- Connected Databricks to Azure Storage using Azure Key Vault-managed Storage Account credentials for secure authentication.
- Read raw CSV from ADLS Gen2
- Verified schema and row count
- Stored raw dataset in Delta format
- Preserved raw data without transformations

### Output
Bronze Delta Table stored in ADLS Gen2.